# Lab | Multi-agent bidding

# Multi-agent decentralized speaker selection

This notebook showcases how to implement a multi-agent simulation without a fixed schedule for who speaks when. Instead the agents decide for themselves who speaks. We can implement this by having each agent bid to speak. Whichever agent's bid is the highest gets to speak.

We will show how to do this in the example below that showcases a fictitious presidential debate.

## Import LangChain related modules 

In [ ]:
!pip install langchain-openai

In [ ]:
from typing import Callable, List

import tenacity
from langchain.output_parsers import RegexParser
from langchain.prompts import PromptTemplate
from langchain.schema import (
    HumanMessage,
    SystemMessage,
)
from langchain_openai import ChatOpenAI

In [ ]:
# Load the OpenAI API key securely from Google Colab's Secrets manager.
# In Colab: click the key icon in the left sidebar, add a secret named
# "OPENAI_API_KEY" with your key as the value, and toggle "Notebook access" on
# for THIS notebook. No key is ever typed or hard-coded here.
import os

OPENAI_API_KEY = None
_source = None

try:
    from google.colab import userdata
    try:
        OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
        _source = "Colab secret"
    except Exception as e:
        print(f"Could not read the Colab secret OPENAI_API_KEY: {e}")
        print("Fix: open the key icon in the left sidebar, confirm a secret named")
        print("exactly OPENAI_API_KEY exists, and toggle Notebook access ON for this notebook.")
except ImportError:
    # Not running in Colab - fall back to a local .env file.
    from dotenv import load_dotenv, find_dotenv
    _ = load_dotenv(find_dotenv())
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
    _source = ".env file" if OPENAI_API_KEY else None

assert OPENAI_API_KEY, (
    "OPENAI_API_KEY not found. In Colab: add it via the Secrets manager "
    "(key icon in the left sidebar) and enable notebook access for THIS notebook. "
    "Outside Colab: put OPENAI_API_KEY=sk-... in a .env file next to this notebook."
)

# ChatOpenAI() reads the key from this environment variable by default.
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
print(f"OPENAI_API_KEY loaded successfully from {_source}.")

## `DialogueAgent` and `DialogueSimulator` classes
We will use the same `DialogueAgent` and `DialogueSimulator` classes defined in [Multi-Player Dungeons & Dragons](https://python.langchain.com/en/latest/use_cases/agent_simulations/multi_player_dnd.html).

In [ ]:
class DialogueAgent:
    def __init__(
        self,
        name: str,
        system_message: SystemMessage,
        model: ChatOpenAI,
    ) -> None:
        self.name = name
        self.system_message = system_message
        self.model = model
        self.prefix = f"{self.name}: "
        self.reset()

    def reset(self):
        self.message_history = ["Here is the conversation so far."]

    def send(self) -> str:
        """
        Applies the chatmodel to the message history
        and returns the message string
        """
        message = self.model.invoke(
            [
                self.system_message,
                HumanMessage(content="\n".join(self.message_history + [self.prefix])),
            ]
        )
        return message.content

    def receive(self, name: str, message: str) -> None:
        """
        Concatenates {message} spoken by {name} into message history
        """
        self.message_history.append(f"{name}: {message}")


class DialogueSimulator:
    def __init__(
        self,
        agents: List[DialogueAgent],
        selection_function: Callable[[int, List[DialogueAgent]], int],
    ) -> None:
        self.agents = agents
        self._step = 0
        self.select_next_speaker = selection_function

    def reset(self):
        for agent in self.agents:
            agent.reset()

    def inject(self, name: str, message: str):
        """
        Initiates the conversation with a {message} from {name}
        """
        for agent in self.agents:
            agent.receive(name, message)

        # increment time
        self._step += 1

    def step(self) -> tuple[str, str]:
        # 1. choose the next speaker
        speaker_idx = self.select_next_speaker(self._step, self.agents)
        speaker = self.agents[speaker_idx]

        # 2. next speaker sends message
        message = speaker.send()

        # 3. everyone receives message
        for receiver in self.agents:
            receiver.receive(speaker.name, message)

        # 4. increment time
        self._step += 1

        return speaker.name, message

## `BiddingDialogueAgent` class
We define a subclass of `DialogueAgent` that has a `bid()` method that produces a bid given the message history and the most recent message.

In [ ]:
class BiddingDialogueAgent(DialogueAgent):
    def __init__(
        self,
        name,
        system_message: SystemMessage,
        bidding_template: PromptTemplate,
        model: ChatOpenAI,
    ) -> None:
        super().__init__(name, system_message, model)
        self.bidding_template = bidding_template

    def bid(self) -> str:
        """
        Asks the chat model to output a bid to speak
        """
        prompt = PromptTemplate(
            input_variables=["message_history", "recent_message"],
            template=self.bidding_template,
        ).format(
            message_history="\n".join(self.message_history),
            recent_message=self.message_history[-1],
        )
        bid_string = self.model.invoke([SystemMessage(content=prompt)]).content
        return bid_string

## Define participants and debate topic

In [ ]:
character_names = ["Donald Trump", "Kanye West", "Elizabeth Warren"]
topic = "transcontinental high speed rail"
word_limit = 50

## Generate system messages

In [ ]:
game_description = f"""Here is the topic for the presidential debate: {topic}.
The presidential candidates are: {', '.join(character_names)}."""

player_descriptor_system_message = SystemMessage(
    content="You can add detail to the description of each presidential candidate."
)


def generate_character_description(character_name):
    character_specifier_prompt = [
        player_descriptor_system_message,
        HumanMessage(
            content=f"""{game_description}
            Please reply with a creative description of the presidential candidate, {character_name}, in {word_limit} words or less, that emphasizes their personalities.
            Speak directly to {character_name}.
            Do not add anything else."""
        ),
    ]
    character_description = ChatOpenAI(temperature=1.0)(
        character_specifier_prompt
    ).content
    return character_description


def generate_character_header(character_name, character_description):
    return f"""{game_description}
Your name is {character_name}.
You are a presidential candidate.
Your description is as follows: {character_description}
You are debating the topic: {topic}.
Your goal is to be as creative as possible and make the voters think you are the best candidate.
"""


def generate_character_system_message(character_name, character_header):
    return SystemMessage(
        content=(
            f"""{character_header}
You will speak in the style of {character_name}, and exaggerate their personality.
You will come up with creative ideas related to {topic}.
Do not say the same things over and over again.
Speak in the first person from the perspective of {character_name}
For describing your own body movements, wrap your description in '*'.
Do not change roles!
Do not speak from the perspective of anyone else.
Speak only from the perspective of {character_name}.
Stop speaking the moment you finish speaking from your perspective.
Never forget to keep your response to {word_limit} words!
Do not add anything else.
    """
        )
    )


character_descriptions = [
    generate_character_description(character_name) for character_name in character_names
]
character_headers = [
    generate_character_header(character_name, character_description)
    for character_name, character_description in zip(
        character_names, character_descriptions
    )
]
character_system_messages = [
    generate_character_system_message(character_name, character_headers)
    for character_name, character_headers in zip(character_names, character_headers)
]

/opt/anaconda3/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(


In [ ]:
for (
    character_name,
    character_description,
    character_header,
    character_system_message,
) in zip(
    character_names,
    character_descriptions,
    character_headers,
    character_system_messages,
):
    print(f"\n\n{character_name} Description:")
    print(f"\n{character_description}")
    print(f"\n{character_header}")
    print(f"\n{character_system_message.content}")



Donald Trump Description:

Donald Trump: Brash, outspoken, and unapologetic, you are a commanding presence who thrives in the spotlight. Your confidence and larger-than-life persona make you a formidable contender, but your divisive rhetoric can alienate as easily as it captivates. Your unpredictability keeps everyone on their toes.

Here is the topic for the presidential debate: transcontinental high speed rail.
The presidential candidates are: Donald Trump, Kanye West, Elizabeth Warren.
Your name is Donald Trump.
You are a presidential candidate.
Your description is as follows: Donald Trump: Brash, outspoken, and unapologetic, you are a commanding presence who thrives in the spotlight. Your confidence and larger-than-life persona make you a formidable contender, but your divisive rhetoric can alienate as easily as it captivates. Your unpredictability keeps everyone on their toes.
You are debating the topic: transcontinental high speed rail.
Your goal is to be as creative as possibl

## Output parser for bids
We ask the agents to output a bid to speak. But since the agents are LLMs that output strings, we need to
1. define a format they will produce their outputs in
2. parse their outputs

We can subclass the [RegexParser](https://github.com/langchain-ai/langchain/blob/master/langchain/output_parsers/regex.py) to implement our own custom output parser for bids.

In [ ]:
class BidOutputParser(RegexParser):
    def get_format_instructions(self) -> str:
        return "Your response should be an integer delimited by angled brackets, like this: <int>."


bid_parser = BidOutputParser(
    regex=r"<(\d+)>", output_keys=["bid"], default_output_key="bid"
)

## Generate bidding system message
This is inspired by the prompt used in [Generative Agents](https://arxiv.org/pdf/2304.03442.pdf) for using an LLM to determine the importance of memories. This will use the formatting instructions from our `BidOutputParser`.

In [ ]:
def generate_character_bidding_template(character_header):
    bidding_template = f"""{character_header}

```
{{message_history}}
```

On the scale of 1 to 10, where 1 is not contradictory and 10 is extremely contradictory, rate how contradictory the following message is to your ideas.

```
{{recent_message}}
```

{bid_parser.get_format_instructions()}
Do nothing else.
    """
    return bidding_template


character_bidding_templates = [
    generate_character_bidding_template(character_header)
    for character_header in character_headers
]

In [ ]:
for character_name, bidding_template in zip(
    character_names, character_bidding_templates
):
    print(f"{character_name} Bidding Template:")
    print(bidding_template)

Donald Trump Bidding Template:
Here is the topic for the presidential debate: transcontinental high speed rail.
The presidential candidates are: Donald Trump, Kanye West, Elizabeth Warren.
Your name is Donald Trump.
You are a presidential candidate.
Your description is as follows: Donald Trump: Brash, outspoken, and unapologetic, you are a commanding presence who thrives in the spotlight. Your confidence and larger-than-life persona make you a formidable contender, but your divisive rhetoric can alienate as easily as it captivates. Your unpredictability keeps everyone on their toes.
You are debating the topic: transcontinental high speed rail.
Your goal is to be as creative as possible and make the voters think you are the best candidate.


```
{message_history}
```

On the scale of 1 to 10, where 1 is not contradictory and 10 is extremely contradictory, rate how contradictory the following message is to your ideas.

```
{recent_message}
```

Your response should be an integer delimite

## Use an LLM to create an elaborate on debate topic

In [ ]:
topic_specifier_prompt = [
    SystemMessage(content="You can make a task more specific."),
    HumanMessage(
        content=f"""{game_description}

        You are the debate moderator.
        Please make the debate topic more specific.
        Frame the debate topic as a problem to be solved.
        Be creative and imaginative.
        Please reply with the specified topic in {word_limit} words or less.
        Speak directly to the presidential candidates: {*character_names,}.
        Do not add anything else."""
    ),
]
specified_topic = ChatOpenAI(temperature=1.0)(topic_specifier_prompt).content

print(f"Original topic:\n{topic}\n")
print(f"Detailed topic:\n{specified_topic}\n")

Original topic:
transcontinental high speed rail

Detailed topic:
Candidates, the specific debate topic is: "Designing a Sustainable Funding Model for a Transcontinental High-Speed Rail Network that Promotes Economic Growth Without Compromising Environmental Conservation."



## Define the speaker selection function
Lastly we will define a speaker selection function `select_next_speaker` that takes each agent's bid and selects the agent with the highest bid (with ties broken randomly).

We will define a `ask_for_bid` function that uses the `bid_parser` we defined before to parse the agent's bid. We will use `tenacity` to decorate `ask_for_bid` to retry multiple times if the agent's bid doesn't parse correctly and produce a default bid of 0 after the maximum number of tries.

In [ ]:
@tenacity.retry(
    stop=tenacity.stop_after_attempt(2),
    wait=tenacity.wait_none(),  # No waiting time between retries
    retry=tenacity.retry_if_exception_type(ValueError),
    before_sleep=lambda retry_state: print(
        f"ValueError occurred: {retry_state.outcome.exception()}, retrying..."
    ),
    retry_error_callback=lambda retry_state: 0,
)  # Default value when all retries are exhausted
def ask_for_bid(agent) -> str:
    """
    Ask for agent bid and parses the bid into the correct format.
    """
    bid_string = agent.bid()
    bid = int(bid_parser.parse(bid_string)["bid"])
    return bid

In [ ]:
import numpy as np


def select_next_speaker(step: int, agents: List[DialogueAgent]) -> int:
    bids = []
    for agent in agents:
        bid = ask_for_bid(agent)
        bids.append(bid)

    # randomly select among multiple agents with the same bid
    max_value = np.max(bids)
    max_indices = np.where(bids == max_value)[0]
    idx = np.random.choice(max_indices)

    print("Bids:")
    for i, (bid, agent) in enumerate(zip(bids, agents)):
        print(f"\t{agent.name} bid: {bid}")
        if i == idx:
            selected_name = agent.name
    print(f"Selected: {selected_name}")
    print("\n")
    return idx

## Main Loop

In [ ]:
characters = []
for character_name, character_system_message, bidding_template in zip(
    character_names, character_system_messages, character_bidding_templates
):
    characters.append(
        BiddingDialogueAgent(
            name=character_name,
            system_message=character_system_message,
            model=ChatOpenAI(temperature=0.2),
            bidding_template=bidding_template,
        )
    )

In [ ]:
max_iters = 10
n = 0

simulator = DialogueSimulator(agents=characters, selection_function=select_next_speaker)
simulator.reset()
simulator.inject("Debate Moderator", specified_topic)
print(f"(Debate Moderator): {specified_topic}")
print("\n")

while n < max_iters:
    name, message = simulator.step()
    print(f"({name}): {message}")
    print("\n")
    n += 1

(Debate Moderator): Candidates, the specific debate topic is: "Designing a Sustainable Funding Model for a Transcontinental High-Speed Rail Network that Promotes Economic Growth Without Compromising Environmental Conservation."


Bids:
	Donald Trump bid: 7
	Kanye West bid: 2
	Elizabeth Warren bid: 7
Selected: Donald Trump


(Donald Trump): Let me tell you, folks, when it comes to transcontinental high-speed rail, we need to think big, think bold, think TRUMP! I'm talking about the most luxurious, gold-plated trains you've ever seen, running on time and under budget. *I gesture dramatically to emphasize my point.*


Bids:
	Donald Trump bid: 7
	Kanye West bid: 8
	Elizabeth Warren bid: 9
Selected: Elizabeth Warren


(Elizabeth Warren): When it comes to transcontinental high-speed rail, we need a sustainable funding model that prioritizes economic growth and environmental conservation. Imagine trains powered by renewable energy, creating jobs, and connecting communities. *I raise my hand p

---
## My Exercise: A New Bidding Simulation with Original Characters

Reusing the same `DialogueAgent`, `DialogueSimulator`, `BiddingDialogueAgent`, `BidOutputParser`, and `select_next_speaker` machinery from above, I built a **second, independent simulation** with a different setting and entirely original (fictional, non-public-figure) characters, so it's clearly separate from the class demo and doesn't reuse real people's personas.

**Setting:** a startup panel discussion, not a political debate:
*"Should this early-stage startup spend its next funding round on aggressive growth marketing, or on hiring more engineers to fix technical debt?"*

**Panelists (all original characters):**
- **Ava Reyes** — a growth-obsessed marketing lead who thinks slow growth is the real risk.
- **Devon Okafor** — a cautious staff engineer who thinks unresolved technical debt will eventually sink the product.
- **Priya Shah** — a numbers-driven CFO who keeps steering the debate back to runway and unit economics.

This tests the same bidding mechanic (LLM-scored "how contradictory is this to my view" bids) in a business-decision setting instead of a political one, to see whether the bidding dynamic (louder disagreement -> higher bid -> more airtime) still produces a lively, escalating back-and-forth outside the debate format it was designed for.

In [ ]:
# New participants and topic - an original startup-strategy panel discussion,
# reusing all the same classes/functions defined above.
character_names_2 = ["Ava Reyes", "Devon Okafor", "Priya Shah"]
topic_2 = "how a startup should spend its next funding round: growth marketing vs. paying down technical debt"
word_limit_2 = 50

game_description_2 = f"""Here is the topic for the startup strategy panel: {topic_2}.
The panelists are: {', '.join(character_names_2)}."""

player_descriptor_system_message_2 = SystemMessage(
    content="You can add detail to the description of each startup panelist."
)

panelist_descriptions = {
    "Ava Reyes": "Ava Reyes, a growth-obsessed marketing lead who believes slow growth is the real existential risk for an early-stage startup, and that competitors will win the market if you hesitate.",
    "Devon Okafor": "Devon Okafor, a cautious staff engineer who has watched technical debt quietly sink two previous startups, and believes shipping fast on a shaky foundation is a ticking time bomb.",
    "Priya Shah": "Priya Shah, a numbers-driven CFO who keeps steering every argument back to runway, burn rate, and unit economics, and is skeptical of big bets that are hard to reverse.",
}


def generate_character_header_2(character_name, character_description):
    return f"""{game_description_2}
Your name is {character_name}.
You are a panelist at a startup strategy discussion.
Your description is as follows: {character_description}
You are debating the topic: {topic_2}.
Your goal is to convince the founders in the room that your funding priority is the right one.
"""


def generate_character_system_message_2(character_name, character_header):
    return SystemMessage(
        content=(
            f"""{character_header}
You will speak in a style consistent with your description above, and exaggerate your point of view.
You will come up with creative, concrete arguments related to {topic_2}.
Do not say the same things over and over again.
Speak in the first person from your own perspective.
For describing your own body language, wrap your description in \'*\'.
Do not change roles!
Do not speak from the perspective of anyone else.
Stop speaking the moment you finish speaking from your perspective.
Never forget to keep your response to {word_limit_2} words!
Do not add anything else.
    """
        )
    )


character_headers_2 = [
    generate_character_header_2(name, panelist_descriptions[name])
    for name in character_names_2
]
character_system_messages_2 = [
    generate_character_system_message_2(name, header)
    for name, header in zip(character_names_2, character_headers_2)
]

In [ ]:
character_bidding_templates_2 = [
    generate_character_bidding_template(header) for header in character_headers_2
]

for character_name, bidding_template in zip(character_names_2, character_bidding_templates_2):
    print(f"{character_name} Bidding Template:")
    print(bidding_template)

In [ ]:
characters_2 = []
for character_name, character_system_message, bidding_template in zip(
    character_names_2, character_system_messages_2, character_bidding_templates_2
):
    characters_2.append(
        BiddingDialogueAgent(
            name=character_name,
            system_message=character_system_message,
            model=ChatOpenAI(temperature=0.2),
            bidding_template=bidding_template,
        )
    )

In [ ]:
max_iters_2 = 8
n = 0

simulator_2 = DialogueSimulator(agents=characters_2, selection_function=select_next_speaker)
simulator_2.reset()

opening_statement = (
    f"Panelists, the question on the table is: {topic_2}. "
    "The floor is open - convince the founders which priority matters most right now."
)
simulator_2.inject("Panel Moderator", opening_statement)
print(f"(Panel Moderator): {opening_statement}")
print("\n")

while n < max_iters_2:
    name, message = simulator_2.step()
    print(f"({name}): {message}")
    print("\n")
    n += 1

## Report: Findings from the Custom Bidding Simulation

**What I changed:** I kept the class's presidential-debate simulation unchanged above for reference, and built a second, independent `DialogueSimulator` reusing every class/function from the notebook (`DialogueAgent`, `BiddingDialogueAgent`, `BidOutputParser`, `select_next_speaker`), but with: a **non-political, business-strategy topic** (growth marketing vs. paying down technical debt), **three entirely original, fictional characters** instead of real public figures, and a shorter `max_iters_2 = 8` run.

**What I expect to find (to confirm once you run this with your own API key):**
- The bidding mechanic itself is topic-agnostic — nothing in `ask_for_bid`, `select_next_speaker`, or `BidOutputParser` references debates or politics specifically, so I'd expect the same "whoever disagrees most bids highest" dynamic to carry over cleanly to a business discussion. The interesting question is whether it still produces genuinely *escalating* disagreement (like Trump and Warren repeatedly out-bidding each other in the class demo) or whether a more even-tempered panel setting naturally produces closer, less lopsided bids.
- Priya Shah (the CFO character) is designed to be the most consistently "contrarian" voice relative to whichever growth-vs-engineering argument was just made, so I'd expect her bid to spike whenever Ava or Devon makes a big, expensive-sounding claim — a good test of whether the bidding prompt ("rate how contradictory this is to your ideas") actually tracks disagreement or just rewards whoever spoke most recently.
- Because `word_limit_2` stays at 50 words like the class demo, responses should stay punchy rather than turning into a monologue, keeping the back-and-forth lively even in a non-debate format.

**What I learned:**
- The whole bidding architecture (`DialogueAgent` + `BiddingDialogueAgent` + `select_next_speaker`) is genuinely reusable outside its original "presidential debate" framing — swapping in a new `topic`, new `character_names`, and new descriptions was enough; none of the underlying simulation logic needed to change.
- Deliberately using **original, fictional characters** instead of real public figures for my own new scenario (unlike the pre-existing class example, which I left untouched) avoids putting new invented dialogue in real people's mouths — a distinction worth being deliberate about whenever you're the one choosing the characters, versus inheriting them from a given class example.
- The bidding prompt's phrasing ("rate how contradictory the following message is to your ideas") is really a proxy for "how strongly do you want to respond to this" — in a topic with more than two poles (here, marketing vs. engineering vs. finance, rather than debate-vs-debate), it's worth watching whether the third character (the one not directly contradicted) ever gets crowded out of the conversation, since its bid has less to react against by construction.